# Patch-seq CCF soma coordinates

ETL the excitatory and inhibitory soma-position CSVs from `data/CCF_Patchseq` into project-scoped CCM `SpatialLocation` rows with location type `SOMA`.

Run [the excitatory Patch-seq ETL](../EXC_WNM_patchseq/etl_patchseq_exc_01.ipynb) and [the inhibitory Patch-seq ETL](../INH_patchseq/etl_patchseq_inh_01.ipynb) first, using the same configured output root. This notebook does not register additional cells. Only CSV specimens belonging to their existing dataset are written; other specimens are skipped, and registered cells without a CSV position are left unchanged.

The CSVs were produced by [the SWC extraction notebook](Patchseq_CCF_extract_positions_from_skels.ipynb). Coordinates are already CCF micrometers: x=anterior-to-posterior, y=dorsal-to-ventral, z=subject-left-to-right, matching the existing `ccfv3_um` space. No scaling, axis permutation, reflection, or clipping is applied. `soma_radius` and `n_nodes` are not spatial coordinates and are not imported.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    read_reference_spaces,
    read_spatial_locations,
    write_models,
)
from connects_common_connectivity.io.path_spec import MODEL_TABLE_PATHS
from connects_common_connectivity.models import (
    AnatomicalOrientation,
    LocationType,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    Unit,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/CCF_Patchseq"
OUTPUT_ROOT = Path(get_settings().output_root)
REFERENCE_SPACE_ID = "ccfv3_um"
COORDINATE_COLUMNS = ["ccf_soma_x", "ccf_soma_y", "ccf_soma_z"]
SOURCES = (
    ("visp_exc_morphology", "visp_exc_patchseq", "ccf_soma_pos_exc_ccf_swc.csv"),
    ("visp_patchseq", "visp_inh_patchseq", "ccf_soma_pos_inh_ccf_swc.csv"),
)

for _, _, filename in SOURCES:
    path = DATA_ROOT / filename
    assert path.is_file(), f"Missing local input: {path}"
for table in ("DataItem", "DataItemDataSetAssociation"):
    path = OUTPUT_ROOT / MODEL_TABLE_PATHS[table]
    assert path.is_dir(), f"Missing CCM table: {path}. Run both base Patch-seq ETLs first."

orientation = AnatomicalOrientation(
    anterior_to_posterior=SignedAxis.PLUS_X,
    dorsal_to_ventral=SignedAxis.PLUS_Y,
    subject_left_to_right=SignedAxis.PLUS_Z,
)

## Validate both sources before writing

Read specimen IDs as strings. Reject empty inputs and missing or duplicate IDs. Print overlapping and nonoverlapping specimen counts, then retain only specimens in the registered dataset and validate their coordinates are finite. Sources with no overlap are skipped without writing. Reuse an existing project-scoped CCF space only if its units and anatomical orientation agree, preserving its other metadata and any existing WNM locations.

In [3]:
associations = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItemDataSetAssociation"])
dataitems = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItem"])
positions_by_project = {}
reference_spaces = []
spatial_locations = []

for project_id, dataset_id, filename in SOURCES:
    cohort = associations.filter(
        (pl.col("project_id") == project_id)
        & (pl.col("dataset_id") == dataset_id)
    )
    assert cohort.height > 0, f"No cohort for {dataset_id}; run its base Patch-seq ETL first."
    assert cohort["dataitem_id"].null_count() == 0, f"Null cohort IDs: {dataset_id}"
    assert cohort["dataitem_id"].n_unique() == cohort.height, f"Duplicate cohort IDs: {dataset_id}"
    registered_ids = set(cohort["dataitem_id"].to_list())
    project_ids = set(dataitems.filter(pl.col("project_id") == project_id)["id"].to_list())
    assert registered_ids <= project_ids, f"Unregistered DataItems in {dataset_id} associations."

    positions = pd.read_csv(
        DATA_ROOT / filename,
        usecols=["specimen_id", *COORDINATE_COLUMNS],
        dtype={"specimen_id": str},
        float_precision="round_trip",
    ).rename(columns={"specimen_id": "dataitem_id"})
    assert not positions.empty, f"No soma positions in {filename}"
    assert positions["dataitem_id"].notna().all(), f"Missing specimen IDs in {filename}"
    assert positions["dataitem_id"].str.fullmatch(r"[0-9]+").all(), f"Invalid specimen IDs in {filename}"
    assert positions["dataitem_id"].is_unique, f"Duplicate specimen IDs in {filename}"
    overlapping = positions["dataitem_id"].isin(registered_ids)
    print(
        f"{filename} vs {dataset_id}: {overlapping.sum():,} overlapping specimens retained; "
        f"{(~overlapping).sum():,} nonoverlapping specimens skipped."
    )
    positions = positions.loc[overlapping].sort_values("dataitem_id").reset_index(drop=True)
    if positions.empty:
        print(f"{dataset_id}: no overlapping specimens; no spatial locations will be written.")
        continue
    xyz = positions[COORDINATE_COLUMNS].to_numpy(dtype="float64")
    assert np.isfinite(xyz).all(), f"Missing or non-finite soma coordinates in {filename}"
    positions_by_project[project_id] = positions

    spaces = (
        read_reference_spaces(
            project_id=project_id,
            reference_space_ids=REFERENCE_SPACE_ID,
            output_root=OUTPUT_ROOT,
        )
        if (OUTPUT_ROOT / MODEL_TABLE_PATHS["ReferenceSpace"]).is_dir()
        else pl.DataFrame()
    )
    if spaces.is_empty():
        reference_spaces.append(ReferenceSpace(
            id=REFERENCE_SPACE_ID,
            project_id=project_id,
            name="CCFv3 micrometer coordinates",
            unit=Unit.MICRONS_LENGTH,
            anatomical_orientation=orientation,
        ))
    else:
        assert spaces.height == 1, f"Ambiguous CCF reference space for {project_id}"
        space = spaces.row(0, named=True)
        assert space["unit"] == Unit.MICRONS_LENGTH.value, f"Incompatible CCF units for {project_id}"
        assert space["anatomical_orientation"] is not None, f"Missing CCF orientation for {project_id}"
        for direction, axis in orientation.model_dump(exclude_none=True).items():
            assert space["anatomical_orientation"][direction] == axis, (
                f"Incompatible CCF orientation for {project_id}: {direction}"
            )

    spatial_locations.extend(
        SpatialLocation(
            project_id=project_id,
            dataitem_id=row.dataitem_id,
            reference_space=REFERENCE_SPACE_ID,
            location_type=LocationType.SOMA,
            description=f"Soma position from CCF-registered SWC; source: {filename}.",
            x=float(row.ccf_soma_x), y=float(row.ccf_soma_y), z=float(row.ccf_soma_z),
        )
        for row in positions.itertuples(index=False)
    )
    print(f"{dataset_id}: {len(positions):,} soma positions; {len(registered_ids) - len(positions):,} registered cells without coordinates.")

ccf_soma_pos_exc_ccf_swc.csv vs visp_exc_patchseq: 388 overlapping specimens retained; 21 nonoverlapping specimens skipped.
visp_exc_patchseq: 388 soma positions; 1,140 registered cells without coordinates.
ccf_soma_pos_inh_ccf_swc.csv vs visp_inh_patchseq: 520 overlapping specimens retained; 0 nonoverlapping specimens skipped.
visp_inh_patchseq: 520 soma positions; 2,359 registered cells without coordinates.


## Write spatial locations

Use the configured output root and standard merge behavior. Only missing reference spaces are created. Existing SOMA rows for these specimens in this CCF space are updated; unrelated spatial locations are retained.

In [4]:
if reference_spaces:
    print(write_models(reference_spaces, output_root=OUTPUT_ROOT))
if spatial_locations:
    print(write_models(spatial_locations, output_root=OUTPUT_ROOT))
else:
    print("No overlapping specimens in either source; no spatial locations written.")

WrittenResult(class_name='ReferenceSpace', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_v1.0.1_data_202611_v1/referencespace'), mode='merge_scoped', predicates=('(target."project_id" IS NOT DISTINCT FROM source."project_id") AND target."id" = source."id"',), rows_written=1)
WrittenResult(class_name='SpatialLocation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_v1.0.1_data_202611_v1/spatiallocation'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataitem_id" = source."dataitem_id" AND target."reference_space" = source."reference_space" AND target."location_type" = source."location_type" AND target."project_id" IN (\'visp_exc_morphology\', \'visp_patchseq\')',), rows_written=908)


## Verify stored coordinates

Read back only the imported specimens, so other cells sharing the CCF space (including WNM) are not treated as unexpected rows. Require exact specimen coverage and unchanged coordinate values. Configured dry runs skip read-back verification explicitly.

In [5]:
if get_settings().dry_run:
    print("Dry run: no CCF tables written; stored-coordinate verification skipped.")
else:
    for project_id, positions in positions_by_project.items():
        spaces = read_reference_spaces(
            project_id=project_id,
            reference_space_ids=REFERENCE_SPACE_ID,
            output_root=OUTPUT_ROOT,
        )
        assert spaces.height == 1, f"Expected one CCF reference space for {project_id}"
        space = spaces.row(0, named=True)
        assert space["unit"] == Unit.MICRONS_LENGTH.value
        assert space["anatomical_orientation"] is not None
        for direction, axis in orientation.model_dump(exclude_none=True).items():
            assert space["anatomical_orientation"][direction] == axis
        stored = read_spatial_locations(
            project_id,
            reference_spaces=REFERENCE_SPACE_ID,
            dataitem_ids=positions["dataitem_id"].tolist(),
            location_types=LocationType.SOMA,
            output_root=OUTPUT_ROOT,
        ).sort("dataitem_id")
        assert stored["dataitem_id"].to_list() == positions["dataitem_id"].tolist(), (
            f"Stored CCF rows do not match the CSV specimens for {project_id}"
        )
        np.testing.assert_array_equal(
            stored.select("x", "y", "z").to_numpy(),
            positions[COORDINATE_COLUMNS].to_numpy(dtype="float64"),
        )
        print(f"{project_id}: {stored.height:,} CCF SOMA coordinates verified.")

visp_exc_morphology: 388 CCF SOMA coordinates verified.
visp_patchseq: 520 CCF SOMA coordinates verified.
